# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faisal-0065/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*



I will use Logistic Regression as a simple classification model for the observed content-performance score. The warehouse data does not contain the original trend_direction field, so I will not invent a future trend label. Instead, I use an observed March performance score and divide content into higher- and lower-performance groups using the median. This provides a simple, interpretable model for decision support and comparison with the Week-4 rule.


In [8]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", HF_TOKEN is not None)

HF token loaded: True


In [9]:
from huggingface_hub import hf_hub_download
import pandas as pd

# ============================================
# LOAD FLYRANK MARCH 2026 DATA
# ============================================

if HF_TOKEN is None:
    raise ValueError(
        "HF_TOKEN was not found in Colab Secrets."
    )

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("File downloaded successfully:")
print(file_path)

march_df = pd.read_parquet(file_path)

print("\nMarch dataframe created successfully.")
print("Shape:", march_df.shape)

print("\nColumns:")
print(list(march_df.columns))

print(
    "\nUnique clients:",
    march_df["client_hash_id"].nunique()
)

File downloaded successfully:
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet

March dataframe created successfully.
Shape: (9841378, 30)

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

Unique clients: 55


In [10]:
print("Dataset shape:", march_df.shape)

print("\nAvailable columns:")
print(march_df.columns.tolist())

Dataset shape: (9841378, 30)

Available columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [11]:
print("Rows:", len(march_df))
print("Unique content:", march_df["content_hash_id"].nunique())
print("Unique clients:", march_df["client_hash_id"].nunique())
print(
    "Date range:",
    march_df["report_date"].min(),
    "to",
    march_df["report_date"].max()
)

Rows: 9841378
Unique content: 331437
Unique clients: 55
Date range: 2026-03-01 to 2026-03-31


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


I will use a grouped split by client. Content items from the same client will stay in the same train or test group. This reduces the chance that the model learns client-specific patterns from training data and is then evaluated on the same client. The future outcome is March 25–31, while the features come only from March 1–24.


In [12]:
from sklearn.model_selection import GroupShuffleSplit

# ============================================
# W03 — CLIENT-AWARE VALIDATION
# ============================================

# Use the existing March dataframe
model_df = march_df.copy()

print("Model dataframe shape:", model_df.shape)


# --------------------------------------------
# 1. Define the client grouping
# --------------------------------------------

groups = model_df["client_hash_id"]


# --------------------------------------------
# 2. Define available features
# --------------------------------------------

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events"
]

# Keep only features that actually exist
feature_columns = [
    col for col in feature_columns
    if col in model_df.columns
]

X = model_df[feature_columns].copy()

print("\nFeatures used for split audit:")
print(feature_columns)

print("\nX shape:", X.shape)


# --------------------------------------------
# 3. Client-aware train/test split
# --------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X,
        groups=groups
    )
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()


# --------------------------------------------
# 4. Get client sets
# --------------------------------------------

train_clients = set(
    groups.iloc[train_idx]
)

test_clients = set(
    groups.iloc[test_idx]
)

client_overlap = (
    train_clients.intersection(test_clients)
)


# --------------------------------------------
# 5. Results
# --------------------------------------------

print("\n========================================")
print("CLIENT-AWARE SPLIT AUDIT")
print("========================================")

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining clients:", len(train_clients))
print("Testing clients:", len(test_clients))

print("Client overlap:", len(client_overlap))

if len(client_overlap) == 0:
    print("\nClient overlap check: PASS")
else:
    print("\nClient overlap check: FAIL")

print("\nNo client identifier is used as a predictive feature.")

print("\nW03 validation split completed successfully.")

Model dataframe shape: (9841378, 30)

Features used for split audit:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'scroll_events']

X shape: (9841378, 14)

CLIENT-AWARE SPLIT AUDIT
Training rows: 8935676
Testing rows: 905702

Training clients: 44
Testing clients: 11
Client overlap: 0

Client overlap check: PASS

No client identifier is used as a predictive feature.

W03 validation split completed successfully.


In [13]:
# ============================================
# W03 — DATA LOADING
# ============================================

import os
import glob
import pandas as pd

print("Searching for CSV files...")

csv_files = glob.glob("**/*.csv", recursive=True)

print("\nCSV files found:")
for file in csv_files:
    print(file)

if not csv_files:
    raise FileNotFoundError(
        "No CSV file found in the current Colab environment."
    )

# Look for the FlyRank dataset
dataset_path = None

for file in csv_files:
    if "content_refresh_anonymized" in os.path.basename(file).lower():
        dataset_path = file
        break

# If exact name isn't found, use the first CSV
if dataset_path is None:
    dataset_path = csv_files[0]

print("\nLoading:")
print(dataset_path)

df = pd.read_csv(dataset_path)

print("\nDataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumns:")
print(list(df.columns))

Searching for CSV files...

CSV files found:
sample_data/california_housing_test.csv
sample_data/mnist_train_small.csv
sample_data/california_housing_train.csv
sample_data/mnist_test.csv

Loading:
sample_data/california_housing_test.csv

Dataset loaded successfully.
Rows: 3000
Columns: 9

Columns:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*



The W05 Logistic Regression model performed substantially better than the W04 rule on the same test rows and the same F1 metric. The measured F1 increased from 0.0134 for the W04 baseline to 0.7134 for the W05 model. Precision and recall also improved substantially. This suggests that the learned model captures useful relationships between past performance signals and the observed future click outcome. The result is directional decision-support rather than proof that the model will perform the same way on future months.


In [14]:
# ============================================
# W05 — CREATE MODEL DATASET
# ============================================

import pandas as pd
import numpy as np

# Make sure March data is available
if "march_df" not in globals():
    raise NameError(
        "march_df is not defined. "
        "Run the Hugging Face data-loading cell first."
    )

# Make sure dates are datetime
march_df["report_date"] = pd.to_datetime(march_df["report_date"])

# --------------------------------------------
# Past data: March 1–24
# --------------------------------------------

past_df = (
    march_df[march_df["report_date"] < "2026-03-25"]
    .groupby(["content_hash_id", "client_hash_id"])
    .agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        pageviews=("ga4_pageviews", "sum"),
        sessions=("ga4_sessions", "sum")
    )
    .reset_index()
)

# --------------------------------------------
# Future data: March 25–31
# --------------------------------------------

future_df = (
    march_df[march_df["report_date"] >= "2026-03-25"]
    .groupby(["content_hash_id", "client_hash_id"])
    .agg(
        future_impressions=("gsc_impressions", "sum"),
        future_clicks=("gsc_clicks", "sum")
    )
    .reset_index()
)

# --------------------------------------------
# Combine past features + future outcome
# --------------------------------------------

model_df = past_df.merge(
    future_df,
    on=["content_hash_id", "client_hash_id"],
    how="inner"
)

# --------------------------------------------
# Create future target
# --------------------------------------------

model_df["future_performance"] = (
    model_df["future_clicks"] > 0
).astype(int)

print("========================================")
print("W05 MODEL DATASET")
print("========================================")

print("Model dataframe shape:", model_df.shape)

print("\nColumns:")
print(model_df.columns.tolist())

print("\nFuture performance distribution:")
print(model_df["future_performance"].value_counts())

print("\nTarget proportions:")
print(model_df["future_performance"].value_counts(normalize=True))

W05 MODEL DATASET
Model dataframe shape: (325118, 10)

Columns:
['content_hash_id', 'client_hash_id', 'impressions', 'clicks', 'avg_position', 'pageviews', 'sessions', 'future_impressions', 'future_clicks', 'future_performance']

Future performance distribution:
future_performance
0    286427
1     38691
Name: count, dtype: int64

Target proportions:
future_performance
0    0.880994
1    0.119006
Name: proportion, dtype: float64


In [15]:
from sklearn.model_selection import GroupShuffleSplit

# ============================================
# W05 — CLIENT-AWARE TRAIN/TEST SPLIT
# ============================================

features = [
    "impressions",
    "clicks",
    "avg_position",
    "pageviews",
    "sessions"
]

X = model_df[features].copy()
y = model_df["future_performance"].copy()
groups = model_df["client_hash_id"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of clients:", groups.nunique())

# Client-aware split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

# Check client separation
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

client_overlap = train_clients.intersection(test_clients)

print("\n========================================")
print("CLIENT-AWARE SPLIT")
print("========================================")

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))

print("Client overlap:", len(client_overlap))

if len(client_overlap) == 0:
    print("Client overlap check: PASS")
else:
    print("Client overlap check: FAIL")

X shape: (325118, 5)
y shape: (325118,)
Number of clients: 55

CLIENT-AWARE SPLIT
Training rows: 295230
Testing rows: 29888
Training clients: 44
Testing clients: 11
Client overlap: 0
Client overlap check: PASS


In [17]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

print("Training Logistic Regression...")

model.fit(X_train, y_train)

print("Model trained successfully!")

Training Logistic Regression...
Model trained successfully!


In [18]:
content_df = (
    march_df
    .groupby(["content_hash_id", "report_date"])
    .agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        pageviews=("ga4_pageviews", "sum"),
        sessions=("ga4_sessions", "sum")
    )
    .reset_index()
)

content_df["ctr"] = (
    content_df["clicks"] /
    content_df["impressions"].replace(0, np.nan)
).fillna(0)

content_df["performance_score"] = (
    0.6 * content_df["ctr"].clip(0, 1)
    + 0.4 * (1 / (1 + content_df["impressions"]))
)

threshold = content_df["performance_score"].median()

content_df["performance_class"] = (
    content_df["performance_score"] >= threshold
).astype(int)

print("Dataset shape:", content_df.shape)
print("Date range:", content_df["report_date"].min(), "to", content_df["report_date"].max())

Dataset shape: (9841378, 10)
Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00


In [19]:
print("Missing values:")
print(content_df.isna().sum())

print("\nDataset shape:", content_df.shape)

Missing values:
content_hash_id            0
report_date                0
impressions                0
clicks                     0
avg_position         6230317
pageviews                  0
sessions                   0
ctr                        0
performance_score          0
performance_class          0
dtype: int64

Dataset shape: (9841378, 10)


In [20]:
# Create an observed performance score from March data.
# This is an observed outcome, not a future label.

content_df["performance_score"] = (
    0.6 * content_df["ctr"].clip(0, 1)
    + 0.4 * (1 / (1 + content_df["impressions"]))
)

print("Performance score summary:")
print(content_df["performance_score"].describe())

Performance score summary:
count    9.841378e+06
mean     2.734887e-01
std      1.709281e-01
min      1.070406e-05
25%      5.714286e-02
50%      4.000000e-01
75%      4.000000e-01
max      8.000000e-01
Name: performance_score, dtype: float64


In [21]:
# Create a binary outcome for a simple classification model.
# 1 = higher observed performance score
# 0 = lower observed performance score

threshold = content_df["performance_score"].median()

content_df["performance_class"] = (
    content_df["performance_score"] >= threshold
).astype(int)

print("Classification threshold:", threshold)
print("\nClass distribution:")
print(content_df["performance_class"].value_counts())

Classification threshold: 0.4

Class distribution:
performance_class
1    6234250
0    3607128
Name: count, dtype: int64


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Features available at the decision moment
features = [
    "impressions",
    "clicks",
    "avg_position",
    "pageviews",
    "sessions"
]

X = content_df[features]
y = content_df["performance_class"]

print("Features:", features)
print("X shape:", X.shape)
print("y shape:", y.shape)

Features: ['impressions', 'clicks', 'avg_position', 'pageviews', 'sessions']
X shape: (9841378, 5)
y shape: (9841378,)


In [23]:
# ============================================
# LOAD MARCH 2026 DATA
# ============================================

from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd

# Get Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError(
        "HF_TOKEN not found. Add your Hugging Face token "
        "to Colab Secrets first."
    )

print("HF token loaded: True")


# Download March 2026 parquet file
file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("\nFile downloaded successfully.")


# Load parquet
march_df = pd.read_parquet(file_path)

print("\nMarch dataframe created successfully.")
print("Shape:", march_df.shape)

print("\nColumns:")
print(list(march_df.columns))

print("\nUnique clients:",
      march_df["client_hash_id"].nunique())

HF token loaded: True

File downloaded successfully.

March dataframe created successfully.
Shape: (9841378, 30)

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

Unique clients: 55


In [24]:
# ============================================
# STEP 1 — CHECK HUGGING FACE TOKEN
# ============================================

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", HF_TOKEN is not None)

HF token loaded: True


In [25]:
# ============================================
# STEP 2 — TEST FLYRANK DATASET ACCESS
# ============================================

from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

try:
    info = api.dataset_info(
        "FlyRank/internship-warehouse"
    )

    print("Dataset access: PASS")
    print("Dataset:", info.id)

except Exception as e:
    print("Dataset access: FAILED")
    print(type(e).__name__)
    print(e)

Dataset access: PASS
Dataset: FlyRank/internship-warehouse


In [26]:
# Make predictions on the test data
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("Model results")
print("----------------")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 score :", round(f1, 4))

Model results
----------------
Accuracy : 0.925
Precision: 0.6943
Recall   : 0.7337
F1 score : 0.7134


In [5]:
# ============================================
# W05 — LOAD FLYRANK MARCH 2026 DATA
# ============================================

import pandas as pd
from google.colab import userdata
from huggingface_hub import hf_hub_download

# Get Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN not found. "
        "Add your Hugging Face token to Colab Secrets."
    )

print("HF token loaded: True")


# Download March 2026 dataset
file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("\nFile downloaded successfully:")
print(file_path)


# Load dataset
march_df = pd.read_parquet(file_path)

print("\n========================================")
print("MARCH DATASET LOADED")
print("========================================")

print("Shape:", march_df.shape)

print("Unique clients:",
      march_df["client_hash_id"].nunique())

print("Unique content:",
      march_df["content_hash_id"].nunique())

print("Date range:",
      march_df["report_date"].min(),
      "to",
      march_df["report_date"].max())

HF token loaded: True

File downloaded successfully:
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet

MARCH DATASET LOADED
Shape: (9841378, 30)
Unique clients: 55
Unique content: 331437
Date range: 2026-03-01 to 2026-03-31


In [7]:
# ============================================
# W05 — COMPLETE MODEL DATASET SETUP
# ============================================

import pandas as pd
import numpy as np

# --------------------------------------------
# 1. Check March dataset
# --------------------------------------------

if "march_df" not in globals():
    raise NameError(
        "march_df is not loaded. "
        "Run the Hugging Face data-loading cell first."
    )

march_df["report_date"] = pd.to_datetime(
    march_df["report_date"]
)

print("March dataset:", march_df.shape)


# --------------------------------------------
# 2. Past data — March 1–24
# --------------------------------------------

past_df = (
    march_df[
        march_df["report_date"] < "2026-03-25"
    ]
    .groupby(
        ["content_hash_id", "client_hash_id"]
    )
    .agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        pageviews=("ga4_pageviews", "sum"),
        sessions=("ga4_sessions", "sum")
    )
    .reset_index()
)

print("Past data:", past_df.shape)


# --------------------------------------------
# 3. Future data — March 25–31
# --------------------------------------------

future_df = (
    march_df[
        march_df["report_date"] >= "2026-03-25"
    ]
    .groupby(
        ["content_hash_id", "client_hash_id"]
    )
    .agg(
        future_impressions=("gsc_impressions", "sum"),
        future_clicks=("gsc_clicks", "sum")
    )
    .reset_index()
)

print("Future data:", future_df.shape)


# --------------------------------------------
# 4. Create model_df
# --------------------------------------------

model_df = past_df.merge(
    future_df,
    on=["content_hash_id", "client_hash_id"],
    how="inner"
)

print("Model data:", model_df.shape)


# --------------------------------------------
# 5. Create future target
# --------------------------------------------

# 1 = at least one click during Mar 25–31
# 0 = no clicks during Mar 25–31

model_df["future_performance"] = (
    model_df["future_clicks"] > 0
).astype(int)


# --------------------------------------------
# 6. Final verification
# --------------------------------------------

print("\n========================================")
print("W05 MODEL DATASET READY")
print("========================================")

print("Model dataframe shape:", model_df.shape)

print("\nColumns:")
print(model_df.columns.tolist())

print("\nFuture performance:")
print(model_df["future_performance"].value_counts())

print("\nFuture performance proportions:")
print(
    model_df["future_performance"]
    .value_counts(normalize=True)
)

display(model_df.head())

March dataset: (9841378, 30)
Past data: (325119, 7)
Future data: (331436, 4)
Model data: (325118, 9)

W05 MODEL DATASET READY
Model dataframe shape: (325118, 10)

Columns:
['content_hash_id', 'client_hash_id', 'impressions', 'clicks', 'avg_position', 'pageviews', 'sessions', 'future_impressions', 'future_clicks', 'future_performance']

Future performance:
future_performance
0    286427
1     38691
Name: count, dtype: int64

Future performance proportions:
future_performance
0    0.880994
1    0.119006
Name: proportion, dtype: float64


,content_hash_id,client_hash_id,impressions,clicks,avg_position,pageviews,sessions,future_impressions,future_clicks,future_performance
0,content_000005d4ced12088,client_9958f0a7ae1df715,56,0,71.491176,0.0,0.0,30,0,0
1,content_00001e488b74b799,client_625b6439094e23e4,0,0,NaN,0.0,0.0,0,0,0
2,content_00007bd2985b77c3,client_73cda7b4e4f265ea,33,0,5.776471,0.0,0.0,14,0,0
3,content_00008950670cb6b5,client_def0955f7a377868,0,0,NaN,1.0,1.0,0,0,0
4,content_0000a348850eb1fc,client_3ffa76342f366962,0,0,NaN,1.0,1.0,0,0,0


In [8]:
# Create a useful future outcome
# 1 = received at least one click during March 25-31
# 0 = received no clicks during March 25-31

model_df["future_performance"] = (
    model_df["future_clicks"] > 0
).astype(int)

print("Future performance distribution:")
print(model_df["future_performance"].value_counts())

print("\nClass proportions:")
print(model_df["future_performance"].value_counts(normalize=True))

Future performance distribution:
future_performance
0    286427
1     38691
Name: count, dtype: int64

Class proportions:
future_performance
0    0.880994
1    0.119006
Name: proportion, dtype: float64


In [9]:
features = [
    "impressions",
    "clicks",
    "avg_position",
    "pageviews",
    "sessions"
]

X = model_df[features]
y = model_df["future_performance"]

print("Features:", features)
print("X shape:", X.shape)
print("Target shape:", y.shape)

Features: ['impressions', 'clicks', 'avg_position', 'pageviews', 'sessions']
X shape: (325118, 5)
Target shape: (325118,)


In [11]:
# ============================================
# W05 — FEATURES + CLIENT-AWARE TRAIN/TEST SPLIT
# ============================================

from sklearn.model_selection import GroupShuffleSplit

# Features from March 1–24 only
features = [
    "impressions",
    "clicks",
    "avg_position",
    "pageviews",
    "sessions"
]

# Create X, y and client groups
X = model_df[features].copy()
y = model_df["future_performance"].copy()
groups = model_df["client_hash_id"].copy()

print("Features:", features)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of clients:", groups.nunique())


# ============================================
# CLIENT-AWARE SPLIT
# ============================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()


# ============================================
# CHECK CLIENT SEPARATION
# ============================================

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

client_overlap = train_clients.intersection(
    test_clients
)

print("\n========================================")
print("CLIENT-AWARE TRAIN/TEST SPLIT")
print("========================================")

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))

print("Client overlap:", len(client_overlap))

if len(client_overlap) == 0:
    print("Client overlap check: PASS")
else:
    print("Client overlap check: FAIL")

print("\nTraining target rate:", round(y_train.mean(), 4))
print("Testing target rate:", round(y_test.mean(), 4))

Features: ['impressions', 'clicks', 'avg_position', 'pageviews', 'sessions']
X shape: (325118, 5)
y shape: (325118,)
Number of clients: 55

CLIENT-AWARE TRAIN/TEST SPLIT
Training rows: 295230
Testing rows: 29888
Training clients: 44
Testing clients: 11
Client overlap: 0
Client overlap check: PASS

Training target rate: 0.1182
Testing target rate: 0.1273


In [14]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# ============================================
# FIT MODEL FIRST
# ============================================

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

print("Training Logistic Regression...")

model.fit(X_train, y_train)

print("Model trained successfully!")


# ============================================
# PREDICTIONS
# ============================================

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]


# ============================================
# METRICS
# ============================================

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)
recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)
f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)
roc_auc = roc_auc_score(
    y_test,
    y_prob
)


# ============================================
# RESULTS
# ============================================

print("\nW05 Logistic Regression results")
print("--------------------------------")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 score :", round(f1, 4))
print("ROC-AUC  :", round(roc_auc, 4))

Training Logistic Regression...
Model trained successfully!

W05 Logistic Regression results
--------------------------------
Accuracy : 0.925
Precision: 0.6943
Recall   : 0.7337
F1 score : 0.7134
ROC-AUC  : 0.9328


In [15]:
# Recreate the W04 baseline on exactly the same W05 test rows

baseline_compare = model_df.iloc[test_idx].copy()

# Calculate CTR
baseline_compare["ctr"] = (
    baseline_compare["clicks"] /
    baseline_compare["impressions"].replace(0, np.nan)
).fillna(0)

# W04 baseline scoring logic
ctr_score = 1 - baseline_compare["ctr"].clip(upper=1)

impression_score = (
    1 / (1 + baseline_compare["impressions"])
)

baseline_compare["baseline_score"] = (
    0.6 * ctr_score +
    0.4 * impression_score
)

# Use the median score as the baseline decision threshold
baseline_threshold = baseline_compare["baseline_score"].median()

baseline_pred = (
    baseline_compare["baseline_score"] >= baseline_threshold
).astype(int)

print("Baseline test rows:", len(baseline_compare))
print("Baseline threshold:", baseline_threshold)
print("Baseline positive predictions:", baseline_pred.sum())

Baseline test rows: 29888
Baseline threshold: 0.7333333333333333
Baseline positive predictions: 15298


In [16]:
# Evaluate the W04 baseline on the same test target

baseline_f1 = f1_score(
    y_test,
    baseline_pred,
    zero_division=0
)

baseline_precision = precision_score(
    y_test,
    baseline_pred,
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    baseline_pred,
    zero_division=0
)

print("W04 Baseline results")
print("--------------------")
print("Precision:", round(baseline_precision, 4))
print("Recall   :", round(baseline_recall, 4))
print("F1 score :", round(baseline_f1, 4))

W04 Baseline results
--------------------
Precision: 0.0084
Recall   : 0.0336
F1 score : 0.0134


In [17]:
# Final W04 vs W05 comparison

comparison = pd.DataFrame({
    "Method": [
        "W04 Baseline",
        "W05 Logistic Regression"
    ],
    "Precision": [
        baseline_precision,
        precision
    ],
    "Recall": [
        baseline_recall,
        recall
    ],
    "F1": [
        baseline_f1,
        f1
    ]
})

display(comparison)

,Method,Precision,Recall,F1
0,W04 Baseline,0.008367,0.033649,0.013402
1,W05 Logistic Regression,0.694279,0.733701,0.713446


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*



The model's errors show 1,229 false positives and 1,013 false negatives on the test set. False positives are content items that the model predicted would receive a future click but did not. False negatives are content items that received a future click but the model did not predict them.

The largest model coefficient is for clicks, followed by impressions and pageviews. This suggests that the model relies strongly on previous search activity and traffic volume when estimating whether content will receive a future click. Average position has a comparatively small coefficient in this model.

The false-positive examples show that some content had strong historical signals but did not receive a future click. The false-negative examples show the opposite: some content with weak or even zero historical clicks still received a future click. This indicates that past traffic signals do not fully explain future content performance.

The model is therefore useful as directional decision-support, but it should not be treated as a guarantee. A named limitation is that the target is based only on whether at least one future click occurred during the seven-day test window, so it does not capture the magnitude or quality of future traffic.


In [18]:
# Create an error analysis table

error_analysis = X_test.copy()

error_analysis["actual"] = y_test.values
error_analysis["predicted"] = y_pred

error_analysis["error_type"] = np.select(
    [
        (error_analysis["actual"] == 1) & (error_analysis["predicted"] == 1),
        (error_analysis["actual"] == 0) & (error_analysis["predicted"] == 0),
        (error_analysis["actual"] == 0) & (error_analysis["predicted"] == 1),
        (error_analysis["actual"] == 1) & (error_analysis["predicted"] == 0)
    ],
    [
        "True Positive",
        "True Negative",
        "False Positive",
        "False Negative"
    ],
    default="Unknown"
)

print("Error counts:")
print(error_analysis["error_type"].value_counts())

print("\nSample false positives:")
display(
    error_analysis[
        error_analysis["error_type"] == "False Positive"
    ].head(10)
)

print("\nSample false negatives:")
display(
    error_analysis[
        error_analysis["error_type"] == "False Negative"
    ].head(10)
)

Error counts:
error_type
True Negative     24855
True Positive      2791
False Positive     1229
False Negative     1013
Name: count, dtype: int64

Sample false positives:


,impressions,clicks,avg_position,pageviews,sessions,actual,predicted,error_type
212,2650,1,3.759422,1.0,1.0,0,1,False Positive
495,172,3,7.935538,3.0,3.0,0,1,False Positive
586,859,6,5.398034,1.0,1.0,0,1,False Positive
926,1094,2,8.302571,1.0,1.0,0,1,False Positive
928,906,3,21.901318,1.0,1.0,0,1,False Positive
980,3592,1,42.315605,0.0,0.0,0,1,False Positive
987,873,2,3.102424,1.0,1.0,0,1,False Positive
1334,1069,4,6.296931,5.0,5.0,0,1,False Positive
1447,2029,4,5.507170,9.0,6.0,0,1,False Positive
1545,476,2,10.673482,3.0,2.0,0,1,False Positive



Sample false negatives:


,impressions,clicks,avg_position,pageviews,sessions,actual,predicted,error_type
83,640,1,7.223065,2.0,1.0,1,0,False Negative
108,990,0,7.060409,0.0,0.0,1,0,False Negative
524,16,0,3.233333,3.0,3.0,1,0,False Negative
575,636,1,6.674342,0.0,0.0,1,0,False Negative
610,1020,0,4.951686,0.0,0.0,1,0,False Negative
782,434,1,16.789944,2.0,2.0,1,0,False Negative
1365,191,0,28.429011,0.0,0.0,1,0,False Negative
1411,600,1,16.720712,2.0,2.0,1,0,False Negative
1924,49,0,4.255456,0.0,0.0,1,0,False Negative
1973,382,0,32.106189,2.0,2.0,1,0,False Negative


In [19]:
# Inspect which features the Logistic Regression model relies on

classifier = model.named_steps["classifier"]

feature_importance = pd.DataFrame({
    "feature": features,
    "coefficient": classifier.coef_[0]
})

feature_importance["absolute_coefficient"] = (
    feature_importance["coefficient"].abs()
)

feature_importance = feature_importance.sort_values(
    "absolute_coefficient",
    ascending=False
)

print("Model feature coefficients:")
display(feature_importance)

Model feature coefficients:


,feature,coefficient,absolute_coefficient
1,clicks,8.409761,8.409761
0,impressions,2.088473,2.088473
3,pageviews,2.036874,2.036874
4,sessions,-1.894826,1.894826
2,avg_position,-0.147043,0.147043


In [20]:
# ============================================
# W05 — LEAKAGE CHECK
# ============================================

future_columns = [
    "future_impressions",
    "future_clicks",
    "future_ctr",
    "future_performance"
]

# Features actually supplied to the model
used_features = set(features)

# Check whether any future/label-derived column
# was accidentally used as a model feature
leaked_features = used_features.intersection(
    future_columns
)

print("========================================")
print("LEAKAGE CHECK")
print("========================================")

print("\nModel features:")
print(features)

print("\nFuture/label-derived columns checked:")
print(future_columns)

print("\nLeaked features used by model:")
print(leaked_features)

if len(leaked_features) == 0:
    print("\nLeakage check: PASS")
    print(
        "The model uses only past-performance features "
        "and does not use future outcome columns."
    )
else:
    print("\nLeakage check: FAIL")
    print(
        "One or more future/label-derived columns "
        "were included as model features."
    )

LEAKAGE CHECK

Model features:
['impressions', 'clicks', 'avg_position', 'pageviews', 'sessions']

Future/label-derived columns checked:
['future_impressions', 'future_clicks', 'future_ctr', 'future_performance']

Leaked features used by model:
set()

Leakage check: PASS
The model uses only past-performance features and does not use future outcome columns.
